# RMSProp 优化算法深度解析



## 0. 基本档案

| 项目 | 内容 |
|---|---|
| **全称** | Root Mean Square Propagation |
| **中文译名** | 均方根传播 |
| **提出者** | Geoffrey Hinton |
| **提出年份** | 2012年 |
| **发表形式** | 课程讲义（Neural Networks for Machine Learning, Coursera） |
| **学术简称** | RMSProp |
| **深度学习社区常用名** | RMSprop / RMSProp（大小写混用，含义相同） |
| **所属家族** | 自适应学习率算法（逐参数独立调整步长） |
| **核心创新** | 用指数加权移动平均替代AdaGrad的历史累加，使学习率不再单调衰减 |
| **理论收敛性** | 非凸条件下能以 $\mathcal{O}(\epsilon^{-4})$ 复杂度收敛到 $\epsilon$-稳定点；特定退化的多项式函数上可实现全局线性收敛 |
| **主要局限** | 对初始学习率仍有一定敏感性；不含动量项，在极端陡峭地形中可能震荡；未解决Adam中后期可能出现的泛化性能退化问题 |

## 1. 算法沿革

RMSProp 是为了解决 **AdaGrad** 算法"学习率过早衰减"的问题而提出的改进版。

### 1.1 演化脉络
- **前辈 (AdaGrad)**：累积所有历史梯度的平方和来调整学习率。
    - **缺陷**：学习率在训练后期单调递减至无限小，模型无法继续学习。
- **RMSProp 的核心改进**：将"历史累加"替换为 **指数加权移动平均 (Exponential Moving Average)**。
    - **效果**：赋予近期梯度更高权重，使学习率能够基于近期变化动态调整，而非单调衰减。
- **后裔 (Adam)**：Adam 可以看作是 **RMSProp + 动量法 (Momentum)** 的结合体。
    - RMSProp 负责为每个参数提供自适应学习率，动量法则负责平滑更新方向。




### 1.2 引例
为了直观展示 RMSProp 和 AdaGrad 的优化行为差异，本节使用 **Beale 函数** 作为测试基准。Beale 函数是优化领域中一个经典的**非凸测试函数**，其数学形式为：

$$
f(x, y) = (1.5 - x + xy)^2 + (2.25 - x + xy^2)^2 + (2.625 - x + xy^3)^2
$$

**可导性分析**：
- Beale 函数由三个二次项（平方项）的线性组合构成，每个平方项内部均为关于 $x$ 和 $y$ 的多项式（最高次为 $xy^3$，即关于 $y$ 的三次项）。
- 从数学结构上看，该函数是**处处连续且无限可微**（$C^\infty$）的，因为：
  - 多项式函数在其定义域 $\mathbb{R}^2$ 上处处光滑可导。
  - 平方运算和加法运算不破坏可微性。
  - 复合函数保持可微性。
- **解析梯度存在且易于计算**，这使其非常适合用于梯度下降类优化算法的仿真和可视化。其梯度分量为：

$$
\begin{aligned}
\frac{\partial f}{\partial x} &= 2(1.5 - x + xy)(-1 + y) + 2(2.25 - x + xy^2)(-1 + y^2) + 2(2.625 - x + xy^3)(-1 + y^3) \\
\frac{\partial f}{\partial y} &= 2(1.5 - x + xy)x + 4(2.25 - x + xy^2)xy + 6(2.625 - x + xy^3)xy^2
\end{aligned}
$$

上述梯度表达式在代码中直接实现（`beale_gradient` 函数），无需数值差分，确保了梯度计算的精确性和仿真结果的可靠性。

**地形特征**：
- **全局最优解**：位于 $(x, y) = (3.0, 0.5)$，最小值为 $f(3.0, 0.5) = 0$。
- **地形结构**：函数呈现**狭长山谷**状，山谷从右上方向左下方延伸。
- **平坦区域**：在 $x \in [-1, 1], y \in [-1, 1]$ 区域以及 $x \approx 3.5, y \approx -2.0$ 附近存在梯度较小的平坦区域，这些区域对优化器构成挑战——梯度过小会导致参数更新缓慢，甚至陷入停滞。
- **陡峭山谷**：山谷两侧壁面陡峭，梯度方向变化剧烈，容易导致优化路径震荡。
- **多重局部结构**：虽然 Beale 函数只有一个全局最优解，但其地形包含多个"准平坦"的次优区域，考验优化器逃离能力。

**选择 Beale 函数的理由**：
1. Beale 函数的非凸性和复杂地形（平坦区 + 陡峭谷）高度模拟了深度学习中常见的损失面特征（鞍点、陡坡、峡谷）。
2. 该函数仅有 2 个变量，便于在二维平面上可视化优化轨迹，直观对比 RMSProp 和 AdaGrad 的行为差异。
3. 解析梯度可精确计算，避免了数值差分带来的误差，使算法对比更纯粹地反映优化器本身的性能差异。
4. 通过观察两种算法在相同起点 $(3.5, 3.0)$ 出发的轨迹和收敛速度，可以清晰展示 RMSProp 如何修正 AdaGrad 的缺陷。

下面的代码将：
- 实现 Beale 函数及其解析梯度；
- 封装 RMSProp、AdaGrad 和标准 GD 的优化步骤；
- 运行 500 步迭代，记录优化轨迹和损失历史；
- 在后续可视化中展示等高线地形、优化路径及损失收敛曲线。

In [ ]:
# =============================================================================
# 导入必要的库
# numpy：数值计算，数组、梯度、网格生成
# plotly.graph_objects：交互式绘图对象，绘制等高线、轨迹、收敛曲线
# plotly.io：控制绘图模板
# =============================================================================
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import plotly.subplots as make_subplots

# 设置plotly全局模板为白底风格
pio.templates.default = "plotly_white"

# =============================================================================
# 1. 定义Beale函数及其解析梯度
# Beale函数：经典非凸测试函数，全局最优(3.0, 0.5)，函数值=0；存在鞍点(0,0)
# =============================================================================
def beale_function(x, y):
    """
    Beale目标函数
    f(x,y) = (1.5 - x + x*y)^2 + (2.25 - x + x*y**2)^2 + (2.625 - x + x*y**3)^2
    """
    term1 = (1.5 - x + x * y)
    term2 = (2.25 - x + x * y**2)
    term3 = (2.625 - x + x * y**3)
    return term1**2 + term2**2 + term3**2


def beale_gradient(x, y):
    """
    Beale函数解析梯度，返回 [df/dx, df/dy]
    """
    term1 = 1.5 - x + x*y
    term2 = 2.25 - x + x*y**2
    term3 = 2.625 - x + x*y**3

    # 对x求偏导
    grad_x = 2*term1*(-1 + y) + 2*term2*(-1 + y**2) + 2*term3*(-1 + y**3)
    # 对y求偏导
    grad_y = 2*term1*x + 4*term2*x*y + 6*term3*x*y**2

    return np.array([grad_x, grad_y])

# =============================================================================
# 2. 优化器单步更新 & 完整迭代流程封装（带详细打印和步长记录）
# optimizer_step：单步参数更新，支持rmsprop / adagrad / vanilla gd
# run_optimization：完整迭代循环，记录轨迹、损失、步长，控制台打印迭代信息
# =============================================================================
def optimizer_step(params, grad, lr, optimizer_state, optimizer_type='rmsprop', beta=0.9, epsilon=1e-8):
    """
    执行单步优化器更新
    params: 当前参数向量 [x, y]
    grad: 当前梯度向量
    lr: 学习率
    optimizer_state: 优化器历史状态字典(v/G)
    optimizer_type: rmsprop / adagrad / else(普通GD)
    beta: RMSProp平滑系数
    epsilon: 防止除零小常数
    return: 更新后参数，更新后的优化器状态字典，有效步长
    """
    if optimizer_type == 'rmsprop':
        # 获取历史二阶动量v，无则初始化为全0
        v = optimizer_state.get('v', np.zeros_like(params))
        # RMSProp二阶动量平滑：指数移动平均梯度平方
        v = beta * v + (1 - beta) * grad**2
        # 参数更新：梯度除以梯度平方移动平均开根号
        effective_lr = lr / (np.sqrt(v) + epsilon)
        params_new = params - effective_lr * grad
        return params_new, {'v': v}, effective_lr

    elif optimizer_type == 'adagrad':
        # 获取累计梯度平方G，无则初始化为全0
        G = optimizer_state.get('G', np.zeros_like(params))
        # AdaGrad：梯度平方持续累加
        G = G + grad**2
        # 参数更新：梯度除以累计梯度平方开根号
        effective_lr = lr / (np.sqrt(G) + epsilon)
        params_new = params - effective_lr * grad
        return params_new, {'G': G}, effective_lr

    else:
        # 普通梯度下降，无状态
        effective_lr = np.full_like(params, lr)
        return params - lr * grad, {}, effective_lr


def run_optimization(optimizer_type, lr=0.01, n_steps=300, start_point=None, beta=0.9, verbose=True):
    """
    完整运行优化迭代流程
    :param optimizer_type: 'rmsprop' / 'adagrad'
    :param lr: 学习率
    :param n_steps: 总迭代步数
    :param start_point: 迭代起始坐标 [x,y]
    :param beta: rmsprop平滑系数
    :param verbose: 是否控制台打印迭代详情
    :return: trajectory所有迭代点数组, loss_history每一步损失数组, lr_history每一步有效步长数组
    """
    # 默认迭代起点
    if start_point is None:
        start_point = np.array([3.5, 3.0])
    params = start_point.copy()

    # 保存迭代轨迹与损失历史，初始存入起点
    trajectory = [params.copy()]
    loss_history = [beale_function(params[0], params[1])]
    # 保存有效步长历史
    lr_history = [np.array([lr, lr])]  # 初始步长

    # 优化器状态字典，保存v/G
    optimizer_state = {}

    # verbose模式打印表头信息
    if verbose:
        print(f"\n{'='*80}")
        print(f"  {optimizer_type.upper()} 优化详细迭代过程")
        print(f"{'='*80}")
        print(f"起始点: ({params[0]:.6f}, {params[1]:.6f})")
        print(f"初始损失: {loss_history[0]:.6e}")
        print(f"学习率: {lr}")
        print(f"总迭代步数: {n_steps}")
        print(f"{'='*80}")
        print(f"{'步数':>6} | {'x':>12} | {'y':>12} | {'损失值':>14} | {'梯度范数':>12} | {'有效步长信息'}")
        print(f"{'-'*80}")

    # 迭代主循环
    for step in range(n_steps):
        # 计算当前位置梯度
        grad = beale_gradient(params[0], params[1])
        grad_norm = np.linalg.norm(grad)

        # 保存更新前参数
        params_old = params.copy()

        # 调用单步更新函数，得到新参数、更新后的状态和有效步长
        params, optimizer_state, effective_lr = optimizer_step(
            params, grad, lr, optimizer_state, optimizer_type, beta
        )

        # 计算更新后损失，记录轨迹、损失和步长
        loss_new = beale_function(params[0], params[1])
        trajectory.append(params.copy())
        loss_history.append(loss_new)
        lr_history.append(effective_lr.copy())

        # verbose模式打印迭代信息
        if verbose:
            if optimizer_type == 'rmsprop':
                v = optimizer_state['v']
                lr_info = f"lr_x={effective_lr[0]:.4e}, lr_y={effective_lr[1]:.4e}"
            elif optimizer_type == 'adagrad':
                G = optimizer_state['G']
                lr_info = f"lr_x={effective_lr[0]:.4e}, lr_y={effective_lr[1]:.4e}"
            else:
                lr_info = f"lr={lr:.4f}"

            # 打印策略：前20步每步打印，后续每10步打印一次
            if step < 20:
                print(f"{step:6d} | {params[0]:12.6f} | {params[1]:12.6f} | {loss_new:14.6e} | {grad_norm:12.6e} | {lr_info[:30]}")
            elif step < n_steps - 1 and step % 10 == 0:
                print(f"{step:6d} | {params[0]:12.6f} | {params[1]:12.6f} | {loss_new:14.6e} | {grad_norm:12.6e} | {lr_info[:30]}")

    # 迭代结束，打印最终状态
    if verbose:
        final_grad = beale_gradient(params[0], params[1])
        print(f"{n_steps-1:6d} | {params[0]:12.6f} | {params[1]:12.6f} | {loss_history[-1]:14.6e} | {np.linalg.norm(final_grad):12.6e} | 最终")
        print(f"{'='*80}")
        print(f"  {optimizer_type.upper()} 优化完成!")
        print(f"  最终位置: ({params[0]:.8f}, {params[1]:.8f})")
        print(f"  最终损失: {loss_history[-1]:.6e}")
        print(f"  最终梯度范数: {np.linalg.norm(final_grad):.6e}")
        print(f"{'='*80}\n")

    return np.array(trajectory), np.array(loss_history), np.array(lr_history)

# =============================================================================
# 3. 执行优化仿真（先打印详细过程，再出图）
# 设置迭代步数、起点，分别运行RMSProp、AdaGrad
# =============================================================================
n_steps = 500
start = np.array([3.5, 3.0])

print("\n" + "="*80)
print("                    Beale 函数优化对比实验")
print("="*80)
print("Beale 函数: f(x,y) = (1.5 - x + xy)² + (2.25 - x + xy²)² + (2.625 - x + xy³)²")
print("全局最优解: (3.0, 0.5), 最优值: 0")
print("="*80)

# 运行 RMSProp 优化（详细打印）
print("\n" + "█"*40 + " RMSProp 优化 " + "█"*40)
traj_rms, loss_rms, lr_rms = run_optimization('rmsprop', lr=0.01, n_steps=n_steps, start_point=start, verbose=True)

# 运行 AdaGrad 优化（详细打印）
print("\n" + "█"*40 + " AdaGrad 优化 " + "█"*40)
traj_adagrad, loss_adagrad, lr_adagrad = run_optimization('adagrad', lr=0.05, n_steps=n_steps, start_point=start, verbose=True)

# =============================================================================
# 4. 生成Beale函数等高线网格数据（不同视角）
# 全景范围：大视野观察整体地形
# 放大区域：根据两条优化轨迹自动计算包围盒，聚焦迭代路径
# Z_log = log10(f+1e‑10)，压缩巨大的数值范围便于可视化
# =============================================================================
# 全景范围网格
x_range_full = np.linspace(-4.5, 4.5, 400)
y_range_full = np.linspace(-4.5, 4.5, 400)
X_full, Y_full = np.meshgrid(x_range_full, y_range_full)
# 计算每个网格点函数值
Z_full = np.array([[beale_function(x, y) for x in x_range_full] for y in y_range_full])
# log10变换，加极小值避免log10(0)
Z_full_log = np.log10(Z_full + 1e-10)

# 放大区域：取两条轨迹的坐标极值
x_min = min(np.min(traj_rms[:, 0]), np.min(traj_adagrad[:, 0]))
x_max = max(np.max(traj_rms[:, 0]), np.max(traj_adagrad[:, 0]))
y_min = min(np.min(traj_rms[:, 1]), np.min(traj_adagrad[:, 1]))
y_max = max(np.max(traj_rms[:, 1]), np.max(traj_adagrad[:, 1]))

# 计算中心点，扩展margin得到正方形显示窗口，保证等比例
x_center = (x_min + x_max) / 2
y_center = (y_min + y_max) / 2
max_span = max(x_max - x_min, y_max - y_min)
margin = max(0.5, max_span * 0.15)
half_side = max_span / 2 + margin

x_min_sq = x_center - half_side
x_max_sq = x_center + half_side
y_min_sq = y_center - half_side
y_max_sq = y_center + half_side

# 放大窗口网格
x_range_zoom = np.linspace(x_min_sq, x_max_sq, 300)
y_range_zoom = np.linspace(y_min_sq, y_max_sq, 300)
X_zoom, Y_zoom = np.meshgrid(x_range_zoom, y_range_zoom)
Z_zoom = np.array([[beale_function(x, y) for x in x_range_zoom] for y in y_range_zoom])
Z_zoom_log = np.log10(Z_zoom + 1e-10)

# =============================================================================
# 5. 计算鞍点信息，生成以鞍点为基准的等距等高线
# 鞍点(0,0)，打印鞍点函数值与log10损失
# =============================================================================
saddle_value = beale_function(0, 0)
saddle_log = np.log10(saddle_value + 1e-10)
print(f"\n鞍点 (0,0) 处函数值: {saddle_value:.6f}")
print(f"鞍点 log10(损失): {saddle_log:.6f}")

# =============================================================================
# 6. 以鞍点为中心生成等距等高线
# 将鞍点对齐到某一条等高线上，保证两张图等高线逻辑统一
# =============================================================================
# ---------- 全景图等高线配置 ----------
log_min_full = -5
log_max_full = 5
n_contours_full = 50
step_size_full = (log_max_full - log_min_full) / (n_contours_full - 1)

# 调整起始log值，让鞍点刚好落在某一条等高线上
nearest_idx_full = int(round((saddle_log - log_min_full) / step_size_full))
adjusted_log_min_full = saddle_log - nearest_idx_full * step_size_full
log_min_full = adjusted_log_min_full
log_max_full = log_min_full + (n_contours_full - 1) * step_size_full
log_contours_full = np.linspace(log_min_full, log_max_full, n_contours_full)
saddle_idx_full = int(round((saddle_log - log_min_full) / step_size_full))
print(f"\n全景图: 等高线数量={n_contours_full}, 步长={step_size_full:.4f}")

# ---------- 放大图等高线配置：复用全景值域，保证颜色完全对齐 ----------
step_size_zoom = step_size_full

# =============================================================================
# 7. 图1+图2：双子图布局，左侧全景等高线，右侧放大区域等高线+迭代路径
# zmin/zmax两张图统一，起点金色五角星，全局最优红色叉号
# annotations增加子图标题：地形全貌、迭代路径（显式xanchor="center"居中）
# =============================================================================
fig_contour = go.Figure()

# ---------------------- 左子图：全景等高线 ----------------------
fig_contour.add_trace(
    go.Contour(
        x=x_range_full, y=y_range_full, z=Z_full_log,
        colorscale='Viridis',
        contours=dict(
            start=log_min_full,
            end=log_max_full,
            size=step_size_full,
            coloring='heatmap',
            showlabels=False
        ),
        showscale=True,
        zmin=log_min_full,
        zmax=log_max_full,
        opacity=0.9,
        colorbar=dict(
            title='log₁₀(损失)',
            len=0.8,
            x=0.50,  # 调整到正中间空隙处
            xanchor='center',
            y=0.5,
            yanchor='middle',
            tickvals=np.arange(np.ceil(log_min_full), np.floor(log_max_full)+1, 1),
            ticktext=[f'10^{int(v)}' for v in np.arange(np.ceil(log_min_full), np.floor(log_max_full)+1, 1)]
        ),
        showlegend=False,
        hovertemplate='x: %{x:.2f}<br>y: %{y:.2f}<br>log₁₀(损失): %{z:.2f}<extra></extra>',
        xaxis='x1', yaxis='y1'
    )
)

# ---------------------- 右子图：放大区域等高线【颜色值域与全景完全对齐】 ----------------------
fig_contour.add_trace(
    go.Contour(
        x=x_range_zoom, y=y_range_zoom, z=Z_zoom_log,
        colorscale='Viridis',
        contours=dict(
            start=log_min_full,
            end=log_max_full,
            size=step_size_zoom,
            coloring='heatmap',
            showlabels=False
        ),
        zmin=log_min_full,
        zmax=log_max_full,
        showscale=False,
        opacity=0.8,
        showlegend=False,
        hovertemplate='x: %{x:.3f}<br>y: %{y:.3f}<br>log₁₀(损失): %{z:.2f}<extra></extra>',
        xaxis='x2', yaxis='y2'
    )
)

# ---------------------- 绘制两条优化轨迹（右侧放大子图） ----------------------
# RMSProp轨迹：青色实线+圆形标记
fig_contour.add_trace(go.Scatter(x=traj_rms[:, 0], y=traj_rms[:, 1], mode='lines+markers', name='RMSProp',
                                 line=dict(color='cyan', width=2, dash='solid'),
                                 marker=dict(size=3, color='cyan', symbol='circle'),
                                 legendrank=1,
                                 hovertemplate='RMSProp<br>x: %{x:.3f}<br>y: %{y:.3f}<extra></extra>',
                                 xaxis='x2', yaxis='y2'))

# AdaGrad轨迹：品红色虚线+方形标记
fig_contour.add_trace(go.Scatter(x=traj_adagrad[:, 0], y=traj_adagrad[:, 1], mode='lines+markers', name='AdaGrad',
                                 line=dict(color='magenta', width=2, dash='dash'),
                                 marker=dict(size=3, color='magenta', symbol='square'),
                                 legendrank=2,
                                 hovertemplate='AdaGrad<br>x: %{x:.3f}<br>y: %{y:.3f}<extra></extra>',
                                 xaxis='x2', yaxis='y2'))

# ----------------------左子图标记点：全局最优、起点 ----------------------
# 全局最优(3.0,0.5)：红色叉号 cross‑thin
fig_contour.add_trace(go.Scatter(x=[3.0], y=[0.5], mode='markers', name='全局最优',
                                 marker=dict(size=12, color='red', symbol='cross-thin', line=dict(width=2, color='red')),
                                 legendrank=3,
                                 showlegend=True, xaxis='x1', yaxis='y1'))

# 起点(3.5,3.0)：金色五角星 star
fig_contour.add_trace(go.Scatter(x=[start[0]], y=[start[1]], mode='markers', name='起点',
                                 marker=dict(size=12, color='gold', symbol='star', line=dict(width=2, color='orange')),
                                 legendrank=4,
                                 showlegend=True, xaxis='x1', yaxis='y1'))

# ----------------------右子图标记点，不重复图例 ----------------------
# 全局最优：红色叉号
fig_contour.add_trace(go.Scatter(x=[3.0], y=[0.5], mode='markers',
                                 marker=dict(size=10, color='red', symbol='cross-thin', line=dict(width=1.5, color='red')),
                                 showlegend=False, xaxis='x2', yaxis='y2'))
# 起点：金色五角星
fig_contour.add_trace(go.Scatter(x=[start[0]], y=[start[1]], mode='markers',
                                 marker=dict(size=10, color='gold', symbol='star', line=dict(width=1, color='orange')),
                                 showlegend=False, xaxis='x2', yaxis='y2'))

# =============================================================================
# 更新双图布局：严格等宽，将ColorBar留白在中间
# =============================================================================
fig_contour.update_layout(
    title=dict(
        text='函数地形与优化路径',
        font=dict(size=20, color='#2c3e50')
    ),
    width=1600,
    height=650,
    margin=dict(l=10, r=80, t=100, b=40),
    showlegend=True,

    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.05,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=12)
    ),

    hovermode='closest',
    template='plotly_white',

    annotations=[
        dict(
            text="地形全貌",
            x=0.19, y=0.94, xref="paper", yref="paper",
            xanchor="center", showarrow=False, font=dict(size=16)
        ),
        dict(
            text="迭代路径",
            x=0.81, y=0.94, xref="paper", yref="paper",
            xanchor="center", showarrow=False, font=dict(size=16)
        )
    ],

    # ------------------- 核心修改：让左右两图严格等宽 -------------------
    # 左图 domain: [0.01, 0.39] (宽度 0.38)
    xaxis1=dict(domain=[0.01, 0.39], title=dict(text='x', font=dict(size=13)), range=[-4.5, 4.5], showgrid=True, gridcolor='lightgray'),
    yaxis1=dict(domain=[0.1, 1.0], title=dict(text='y', font=dict(size=13)), range=[-4.5, 4.5], showgrid=True, gridcolor='lightgray', scaleanchor='x1', scaleratio=1),

    # 右图 domain: [0.61, 0.99] (宽度 0.38)
    xaxis2=dict(domain=[0.61, 0.99], title=dict(text='x', font=dict(size=13)), range=[x_min_sq, x_max_sq], showgrid=True, gridcolor='lightgray'),
    yaxis2=dict(domain=[0.1, 1.0], title=dict(text='y', font=dict(size=13)), range=[y_min_sq, y_max_sq], showgrid=True, gridcolor='lightgray', scaleanchor='x2', scaleratio=1)
    # ------------------- 核心修改结束 -------------------
)

fig_contour.show()

# =============================================================================
# 8. 图3：损失收敛曲线对比
# 绘制RMSProp、AdaGrad损失随迭代步数变化，叠加理论最优0参考虚线
# =============================================================================
fig_loss = go.Figure()

# RMSProp损失曲线
fig_loss.add_trace(go.Scatter(x=list(range(len(loss_rms))), y=loss_rms, mode='lines+markers', name='RMSProp',
                              line=dict(color='cyan', width=1.5, dash='solid'),
                              marker=dict(size=3, color='cyan', symbol='circle'),
                              hovertemplate='RMSProp<br>步数: %{x}<br>损失: %{y:.2e}<extra></extra>'))

# AdaGrad损失曲线
fig_loss.add_trace(go.Scatter(x=list(range(len(loss_adagrad))), y=loss_adagrad, mode='lines+markers', name='AdaGrad',
                              line=dict(color='magenta', width=1.5, dash='dash'),
                              marker=dict(size=3, color='magenta', symbol='square'),
                              hovertemplate='AdaGrad<br>步数: %{x}<br>损失: %{y:.2e}<extra></extra>'))

# 理论最优损失参考线 y=0
fig_loss.add_trace(go.Scatter(x=[0, n_steps], y=[0, 0], mode='lines', name='理论最优损失 = 0',
                              line=dict(color='green', width=1.5, dash='dash'),
                              hoverinfo='skip'))

fig_loss.update_layout(
    title=dict(text='损失函数收敛曲线对比', font=dict(size=18, color='#2c3e50')),
    width=1500,
    height=500,
    margin=dict(l=10, r=10, t=70, b=10),
    xaxis=dict(
        title='迭代步数',
        range=[0, n_steps],
        gridcolor='lightgray',
        automargin=True,
        showgrid=True
    ),
    yaxis=dict(
        title='损失值',
        type='linear',
        gridcolor='lightgray',
        automargin=True,
        showgrid=True,
        zeroline=True,
        zerolinecolor='lightgray',
        zerolinewidth=1
    ),

    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=12)
    ),

    hovermode='x unified',
    template='plotly_white'
)

fig_loss.show()

# =============================================================================
# 9. 有效步长变化曲线（配色统一：RMSProp=青色，AdaGrad=品红色）
# 展示RMSProp和AdaGrad在优化过程中x、y维度的有效学习率变化
# x方向：实线（solid），y方向：虚线（dash）
# =============================================================================
fig_lr = go.Figure()

# 创建步长数组
steps = np.arange(len(lr_rms))

# RMSProp x方向有效步长（青色实线）
fig_lr.add_trace(go.Scatter(x=steps, y=lr_rms[:, 0], mode='lines', name='RMSProp (x方向)',
                            line=dict(color='cyan', width=2, dash='solid'),
                            hovertemplate='RMSProp x<br>步数: %{x}<br>有效步长: %{y:.4e}<extra></extra>'))

# RMSProp y方向有效步长（青色虚线）
fig_lr.add_trace(go.Scatter(x=steps, y=lr_rms[:, 1], mode='lines', name='RMSProp (y方向)',
                            line=dict(color='cyan', width=2, dash='dash'),
                            hovertemplate='RMSProp y<br>步数: %{x}<br>有效步长: %{y:.4e}<extra></extra>'))

# AdaGrad x方向有效步长（品红色实线）
fig_lr.add_trace(go.Scatter(x=steps, y=lr_adagrad[:, 0], mode='lines', name='AdaGrad (x方向)',
                            line=dict(color='magenta', width=2, dash='solid'),
                            hovertemplate='AdaGrad x<br>步数: %{x}<br>有效步长: %{y:.4e}<extra></extra>'))

# AdaGrad y方向有效步长（品红色虚线）
fig_lr.add_trace(go.Scatter(x=steps, y=lr_adagrad[:, 1], mode='lines', name='AdaGrad (y方向)',
                            line=dict(color='magenta', width=2, dash='dash'),
                            hovertemplate='AdaGrad y<br>步数: %{x}<br>有效步长: %{y:.4e}<extra></extra>'))

# 添加理论初始步长参考线
fig_lr.add_trace(go.Scatter(x=[0, n_steps], y=[0.01, 0.01], mode='lines', name='RMSProp初始lr = 0.01',
                            line=dict(color='gray', width=1, dash='dot'), hoverinfo='skip'))
fig_lr.add_trace(go.Scatter(x=[0, n_steps], y=[0.05, 0.05], mode='lines', name='AdaGrad初始lr = 0.05',
                            line=dict(color='gray', width=1, dash='dot'), hoverinfo='skip'))

fig_lr.update_layout(
    title=dict(text='有效步长变化曲线', font=dict(size=18, color='#2c3e50')),
    width=1500,
    height=600,
    margin=dict(l=10, r=10, t=70, b=10),
    xaxis=dict(
        title='迭代步数',
        range=[0, n_steps],
        gridcolor='lightgray',
        automargin=True,
        showgrid=True
    ),
    yaxis=dict(
        title='有效步长 (α_t)',
        type='log',
        gridcolor='lightgray',
        automargin=True,
        showgrid=True,
        zeroline=False
    ),

    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        bordercolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=12)
    ),

    hovermode='x unified',
    template='plotly_white'
)

fig_lr.show()

# =============================================================================
# 10. 控制台打印量化统计结果
# =============================================================================
print("\n" + "="*80)
print("                      优化结果对比总结")
print("="*80)
print(f"Beale函数最优值为 0，在 (3.0, 0.5) 处")
print(f"鞍点位于 (0, 0)，函数值为 {saddle_value:.6f}")
print("-"*80)
print(f"RMSProp  最终损失: {loss_rms[-1]:.2e}, 位置: ({traj_rms[-1][0]:.8f}, {traj_rms[-1][1]:.8f})")
print(f"AdaGrad  最终损失: {loss_adagrad[-1]:.2e}, 位置: ({traj_adagrad[-1][0]:.8f}, {traj_adagrad[-1][1]:.8f})")
print("-"*80)

threshold = 1e-3
steps_rms = np.argmax(loss_rms < threshold) if np.any(loss_rms < threshold) else len(loss_rms)
steps_adagrad = np.argmax(loss_adagrad < threshold) if np.any(loss_adagrad < threshold) else len(loss_adagrad)

if steps_rms < len(loss_rms):
    print(f"✓ RMSProp  达到 {threshold} 阈值需要 {steps_rms} 步")
else:
    print(f"✗ RMSProp  在 {len(loss_rms)} 步内未达到 {threshold} 阈值")

if steps_adagrad < len(loss_adagrad):
    print(f"✓ AdaGrad  达到 {threshold} 阈值需要 {steps_adagrad} 步")
else:
    print(f"✗ AdaGrad  在 {len(loss_adagrad)} 步内未达到 {threshold} 阈值")

print("="*80)
print("\n📊 有效步长统计:")
print("-"*80)
print(f"RMSProp  最终步长: x方向 {lr_rms[-1][0]:.4e}, y方向 {lr_rms[-1][1]:.4e}")
print(f"AdaGrad  最终步长: x方向 {lr_adagrad[-1][0]:.4e}, y方向 {lr_adagrad[-1][1]:.4e}")
print(f"RMSProp  步长范围: x方向 [{np.min(lr_rms[:,0]):.4e}, {np.max(lr_rms[:,0]):.4e}]")
print(f"AdaGrad  步长范围: x方向 [{np.min(lr_adagrad[:,0]):.4e}, {np.max(lr_adagrad[:,0]):.4e}]")
print("="*80)

print("\n📊 地形特征:")
print("-"*80)
print("• 全局最优: (3.0, 0.5), 损失值 0")
print(f"• 鞍点: (0, 0), 损失值 {saddle_value:.4f}")
print("• 狭长山谷地形，存在平坦区域")
print(f"• 等高线在log10空间完全等距，步长 {step_size_full:.3f}")
print(f"• 穿过鞍点的等高线索引 {saddle_idx_full} / {len(log_contours_full)}")
print("="*80)

由迭代路径及损失函数收敛图可见：

- RMSProp（青色）进入峡谷后存在小幅迂回震荡，整体持续向全局最优点靠近。
- AdaGrad（品红色）前期轨迹与 RMSProp 接近，进入峡谷后迭代前进幅度快速变小，轨迹提前停滞，距离全局最优点存在明显距离

有效步长变化曲线图清晰地展示了完全不同的步长演化策略：

| 优化器 | 初始步长 | 变化趋势 | 最终步长 | 行为特征 |
|--------|----------|----------|----------|----------|
| **RMSProp (青色)** | 0.01 | 先骤降 → 后持续攀升 → 末期震荡 | ~0.1（X方向） | 自适应恢复，后期加速 |
| **AdaGrad (品红色)** | 0.05 | 前20步断崖式下跌 → 彻底躺平 | ~10⁻⁵（X/Y方向） | 永久衰减，完全停滞 |



可见算法本质差异

AdaGrad 的性格：一次惊吓，终身回避
- 最致命的缺陷是**累积项无上限**。初始阶段的大梯度会永久性地压低后续所有步长。
- 这解释了为什么 AdaGrad 在图中的轨迹只能走到半山腰，无法逼近全局最优——它在**前 20 步内就已经完全丧失了继续优化的能力**。
- 
RMSProp 的性格：遗忘过去，灵活应变
- 核心优势是**指数加权滑动平均**，具有“遗忘历史”的特性。
- 陡坡上大梯度时，降低步长以保证稳定；平坦区小梯度时，释放步长以加速收敛。
- 这解释了为什么 RMSProp 能在 500 步内成功穿越峡谷并到达最终最优解。




## 2. RMSProp算法原理

### 从AdaGrad的困境出发

理解RMSProp，首先要回到AdaGrad的"死穴"：AdaGrad将所有历史梯度平方**无差别地累加**（$G_t = G_{t-1} + g_t^2$），分母持续膨胀，导致学习率单调递减直至趋近于零——这在Beale函数的仿真中清晰可见：AdaGrad进入峡谷后步长急剧收缩，最终停滞在半路。

RMSProp的破解之道，就在它的名字里：

- **R**oot **M**ean **S**quare（均方根）—— 分母用的是梯度平方的**均值根号**，而非累加和；
- **P**rop（自适应）—— 这个均方根是**动态更新**的，而非固化累加。

### 核心公式与设计思想

RMSProp 的更新公式如下：

1. **计算梯度平方的指数加权移动平均**：
   $$ v_t = \beta v_{t-1} + (1 - \beta) g_t^2 $$
   - $\beta$ 是衰减率（通常取 0.9），控制历史信息的衰减速度。
   - $g_t$ 是当前梯度。

2. **参数更新**：
   $$ \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t} + \epsilon} g_t $$
   - $\eta$ 是初始学习率。
   - $\epsilon$ 是一个极小常数（如 1e-8），防止分母为零。

**符号说明**：以上公式中的所有运算（平方、开根号、除法）均为**逐元素（element-wise）**操作。假设参数 $\theta$ 是 $n$ 维向量，则 $g_t$、$v_t$ 均为同维度向量：
$$g_t = [g_{t,1}, g_{t,2}, \dots, g_{t,n}]^T, \quad v_t = [v_{t,1}, v_{t,2}, \dots, v_{t,n}]^T$$

每个分量独立更新：
$$v_{t,i} = \beta v_{t-1,i} + (1-\beta) g_{t,i}^2, \quad \theta_{t+1,i} = \theta_{t,i} - \frac{\eta}{\sqrt{v_{t,i}} + \epsilon} g_{t,i}$$

这意味着**每个参数拥有独立的自适应学习率**——这正是"Prop"（自适应）的真正含义。

**这就是"Root Mean Square"的由来**：分母中的 $\sqrt{v_t}$ 正是梯度平方指数加权平均的**平方根**——即"均方根"（RMS），它代表近期梯度幅度的典型大小。

### 设计思想的三个层次

**第一层：用"移动平均"替代"累加"**
$$v_t = \beta v_{t-1} + (1 - \beta) g_t^2 \quad \text{vs.} \quad G_t = G_{t-1} + g_t^2$$
指数加权平均等价于对近期梯度赋予高权重、对遥远历史赋予指数衰减权重。这相当于一个**自适应滑动窗口**——让优化器始终以最近的地形起伏为参照，而非背负全部历史包袱。正是这一改动，使得RMSProp的梯度平方统计量不会无限增长，学习率也得以保持生命力。

**第二层：按梯度尺度"归一化"步长**
$$\text{有效学习率} = \frac{\eta}{\sqrt{v_t}}$$
分母 $\sqrt{v_t}$ 度量的是该参数近期梯度的**典型幅度**。若某方向梯度长期偏大（地形陡峭），$\sqrt{v_t}$ 较大，步长被压缩，防止震荡；若梯度长期偏小（地形平坦），$\sqrt{v_t}$ 较小，步长被放大，加速穿越。**每个参数拥有独立的自适应步长**——这正是"Prop"（自适应）的含义。

**第三层：几何直觉——山谷中的智能步伐**
回到Beale函数的狭长山谷：垂直于谷底的方向梯度极大（陡壁），沿谷底方向梯度极小（缓坡）。RMSProp在陡壁方向压低步长抑制震荡，在缓坡方向放大步长加速前进。相比之下，AdaGrad在陡壁方向积累了巨大的$G$后，**所有方向**（包括缓坡方向）都被过度压制，最终寸步难行。RMSProp的"移动平均"机制则让$v_t$随地形变化而动态调整，陡壁方向的$v$虽大但不至于拖垮其他方向的步长。

### 一句话总结

> **RMSProp = 每个参数独立维护一个"近期梯度均方根"作为步长标尺，让学习率随地形起伏而灵活伸缩，既不像AdaGrad那样不可逆地衰竭，也不像GD那样对所有方向一视同仁。**

## 3. 突出性能表现

RMSProp 在深度学习实践中展现出多方面的卓越性能：

### 3.1. 在非凸问题上表现卓越
深度学习目标函数充满鞍点和陡坡。RMSProp 的灵活性使其能：
- **攻克鞍点**：在梯度平缓时自动增大学习率，冲出不平稳区域。

  当参数进入鞍点或平坦区域时，梯度 $g_t$ 的值变得极小（接近 0），梯度平方 $g_t^2$ 也接近于 0。在 RMSProp 的更新公式 $v_t = \beta v_{t-1} + (1-\beta) g_t^2$ 中，由于 $g_t^2 \approx 0$，第二项 $(1-\beta)g_t^2$ 几乎可以忽略，因此 $v_t \approx \beta v_{t-1}$。这意味着 $v_t$ 会以指数衰减的速度迅速缩小，历史信息 $v_{t-1}$ 被快速“遗忘”。随着 $v_t$ 不断变小，分母 $\sqrt{v_t} + \epsilon$ 也随之减小，有效学习率 $\eta / (\sqrt{v_t} + \epsilon)$ 急剧增大。尽管此时梯度本身很小，但放大了很多倍的有效学习率与之相乘，恰好产生了一个“适中”的步长，赋予了优化器足够的“冲劲”跨越平坦区域的停滞状态，而不是像标准 SGD 那样因为梯度太小而原地踏步。

- **适应陡坡**：在梯度陡峭时迅速降低学习率，防止震荡发散。

  当参数位于损失面陡峭的峡谷壁时，梯度 $g_t$ 的值非常大，梯度平方 $g_t^2$ 会爆炸式增长。根据更新公式 $v_t = \beta v_{t-1} + (1-\beta) g_t^2$，由于 $g_t^2$ 极大，即使 $1-\beta$ 只有 0.1，$(1-\beta)g_t^2$ 仍然是一个巨大的数，使得 $v_t$ 迅速上升，分母 $\sqrt{v_t} + \epsilon$ 急剧膨胀，有效学习率 $\eta / (\sqrt{v_t} + \epsilon)$ 被大幅压缩。虽然原始的梯度 $g_t$ 幅度巨大，但乘以被压得很小的有效学习率后，最终的更新步长被牢牢限制住，参数每次只移动一小步，小心翼翼地沿着陡坡下滑，避免了因跨步太大而越过谷底或在峡谷壁之间来回反弹震荡，保证了训练过程的稳定性。

### 3.2. 对超参数（学习率）容忍度高
相比 GD，RMSProp 对初始学习率的设置不敏感。即使设置较大的学习率（如 0.001），它也能在后期保持稳定收敛，显著降低了调参成本。

### 3.3. 擅长处理非平稳目标和在线学习
在处理 RNN 或 LSTM 等序列模型时，数据分布随时间变化。RMSProp 的"指数加权平均"天然具备对近期变化的敏感性，能够快速适应数据分布的最新动态。

### 3.4. 训练过程更平稳
通过自适应缩放每个参数的学习率，RMSProp 有效抑制了梯度方向上的剧烈震荡，使损失下降曲线更平滑，收敛过程更稳定。

## 4. 总结

- **定位**：RMSProp 是专为"复杂且非凸"的深度学习问题设计的优化器。
- **核心价值**：通过指数加权移动平均实现灵活、鲁棒的自适应学习率调整。
- **历史地位**：它修正了 AdaGrad 的缺陷，其核心思想被 Adam 等后续主流算法所吸收，是优化算法演进中的关键一环。

## 附录

### 附录A. 公式细节补遗

#### A.1 为什么 $g_t^2$ 是逐元素平方？

在 RMSProp 公式中，$g_t^2$ 表示**逐元素平方**（element-wise square），而非向量点积或矩阵乘法。

假设参数 $\theta$ 是 $n$ 维向量：
$$\theta = [\theta_1, \theta_2, \dots, \theta_n]^T$$

其梯度 $g_t$ 也是同维度的向量：
$$g_t = \left[ \frac{\partial L}{\partial \theta_1}, \frac{\partial L}{\partial \theta_2}, \dots, \frac{\partial L}{\partial \theta_n} \right]^T$$

则 $g_t^2$ 的含义为：
$$g_t^2 = [g_{t,1}^2, g_{t,2}^2, \dots, g_{t,n}^2]^T$$

结果**仍然是一个 $n$ 维向量**，而非标量。这保证了每个参数维度拥有独立的 $v_{t,i}$，从而实现逐参数的自适应学习率。

#### A.2 梯度平方后方向信息去哪了？（严谨修正版）

梯度携带两类信息：

| 信息类型 | 作用 | 在RMSProp中的去处 |
|---------|------|------------------|
| **方向**（正/负） | 决定参数该**增大还是减小** | 保留在更新公式的 **$g_t$** 中（分子） |
| **幅度**（大小） | 决定参数该**走多远** | 被平方后用于计算 $v_t$（分母），决定步长 |

公式中，方向信息从未丢失。需要特别注意的是，由于 $v_t$ 是一个向量，分母 $\sqrt{v_t} + \epsilon$ 也是一个向量，因此参数更新使用**逐元素除法**（Hadamard 除法，记作 $\odot$）：

$$ \theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t} + \epsilon} \odot g_t $$

等价地，对每个参数维度 $i$ 单独写为：

$$ \theta_{t+1}^{(i)} = \theta_t^{(i)} - \frac{\eta}{\sqrt{v_t^{(i)}} + \epsilon} \cdot g_t^{(i)} $$

此时，$\eta / (\sqrt{v_t^{(i)}} + \epsilon)$ 是**第 $i$ 个参数独有的有效学习率**，是一个标量；而 $g_t^{(i)}$ 是该维度的梯度分量（包含正负号）。

- **分子上的 $g_t$** 保留了完整的正负号 → **决定往哪个方向走**
- **分母上的 $\sqrt{v_t}$** 只关心梯度的大小（平方后开根）→ **为每个参数独立决定走多远**

两者**分工明确，互不干扰**。$v_t$ 的作用是**估计该参数方向上的梯度典型幅度**（类似标准差），用来归一化步长——这个判断完全不依赖正负号。

> **⚠️ 常见错误警示**：切勿将向量版本的公式误写为 $\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t} + \epsilon} g_t$，其中将分母视为标量。正确的理解是分母为一个向量，与 $g_t$ 做逐元素除法，或等价地写成每个维度分别除以各自的标量值。

#### A.3 为什么 $v_{t-1}$ 不平方？

因为 $v_{t-1}$ 本身已经是"历史梯度平方的加权平均"，它天然带有"平方"的量纲，不需要再平方一次。

从递推关系展开：

$$v_1 = (1-\beta)g_1^2$$
$$v_2 = \beta v_1 + (1-\beta)g_2^2 = \beta(1-\beta)g_1^2 + (1-\beta)g_2^2$$
$$v_3 = \beta v_2 + (1-\beta)g_3^2 = \beta^2(1-\beta)g_1^2 + \beta(1-\beta)g_2^2 + (1-\beta)g_3^2$$

一般形式：
$$v_t = (1-\beta)\sum_{k=1}^{t} \beta^{t-k} g_k^2$$

$v_{t-1}$ 本身就是 $g_1^2, g_2^2, \dots, g_{t-1}^2$ 的加权和——它已经是"平方的量级"了。

如果错误地对 $v_{t-1}$ 也平方：
$$v_t = \beta v_{t-1}^2 + (1-\beta)g_t^2$$

展开后会出现 $g_1^4$ 项，其量纲是（梯度）⁴，而 $g_t^2$ 的量纲是（梯度）²，**两项量纲不同，无法相加**，物理意义完全混乱。

正确的理解是：$v_{t-1}$ 和 $g_t^2$ 具有**完全相同的量纲和语义**——都是"梯度平方的某种统计量"，因此可以直接加权相加。

对比 AdaGrad 看得更清楚：

| 算法 | 更新公式 | 语义 |
|------|---------|------|
| AdaGrad | $G_t = G_{t-1} + g_t^2$ | 把**历史所有梯度平方**逐个累加起来 |
| RMSProp | $v_t = \beta v_{t-1} + (1-\beta)g_t^2$ | 把**历史梯度平方的加权平均**和**当前梯度平方**做加权平均 |

两者的共同点是：**被累加/平均的对象始终是 $g_k^2$，而不是 $v_{k-1}^2$**。

#### A.4 梯度平方后，如何区分"方向震荡"和"方向一致但幅度大"？

平方本身并不区分这两种情况——两者平方后都是大数。真正区分它们的是**更新公式中分母和分子的配合**：

| 场景 | $g_t$（分子） | $\sqrt{v_t}$（分母） | 净效果 |
|------|-------------|-------------------|--------|
| 方向震荡（+2, -2, +2, -2） | 当前符号不确定 | 大 | 每一步都被大幅压缩，**走不动** |
| 方向一致且幅度大（+2, +2, +2, +2） | 正号一致 | 大 | 步长虽被压缩，但方向始终一致，**持续前进** |
| 方向一致但幅度小（+0.1, +0.1, ...） | 小 | 小 | 步长被放大，**加速前进** |

也就是说：
- **震荡方向** → 分子正负交替 × 分母大 → 来回无效步长被抑制
- **一致方向** → 分子符号稳定 × 分母大 → 步长虽小，但叠加起来依然前进



### 附录B. RMSProp的历史由来与理论证明

#### B.1 由来：从启发性改进说起

RMSProp最早由Geoffrey Hinton在**2012年**的课程讲义中提出，其诞生源于对AdaGrad缺陷的一个简单而直接的修补。

**实践驱动的设计**：

Hinton观察到，AdaGrad的学习率单调递减问题在深度学习中尤为致命——模型参数量大、训练周期长，学习率过早衰竭意味着模型无法充分收敛。他的改进方案是：

$$\text{AdaGrad 的累加} \longrightarrow \text{RMSProp 的指数加权移动平均}$$

这个改动并非源于某个严格的数学定理，而是基于**深刻的实践洞察**——让优化器更关注近期梯度变化，从而避免学习率的不可逆衰竭。因此可以说，RMSProp最初是**启发式**（heuristic）的，而非从数学推导中演绎而来。

#### B.2 理论证明的后来完善

尽管RMSProp的提出是启发性的，但学术界后来对其收敛性做了严格的理论分析。

**早期困难**：由于深度神经网络的非凸性，早期对自适应优化算法的收敛性分析往往局限在**凸函数**假设下，这与实际应用场景差距很大。

**近年来的突破**：

1. **非凸优化下的收敛性**：已有研究证明，在非凸优化条件下，RMSProp在合理的超参数设置下，能以 $\mathcal{O}(\epsilon^{-4})$ 的迭代复杂度收敛到 $\epsilon$-稳定点，匹配了理论上的复杂度下界。

2. **特定函数上的全局收敛**：在某些高度退化的多项式函数（如 $L(x)=\frac{1}{k}x^k$）上，已有研究证明了RMSProp能够实现**全局线性收敛**，并在理论层面揭示了其相较于标准梯度下降（GD）在计算复杂度上的显著优势。

#### B.3 总结

RMSProp的诞生是**实践先于理论**的经典案例。它发端于对AdaGrad缺陷的直观修补，经过多年的实践检验后被证明效果卓越，直到近些年学术界才在理论上逐步证明了其在广泛非凸问题中的有效性。这种"实践引领、理论跟进"的模式，在深度学习优化算法的发展史上并不罕见。


### 附录C. RMSProp的最终性能综览

本附录综合整理了RMSProp在各类任务与实验中的最终性能表现，旨在提供一个全局性的性能画像。

#### C.1 收敛速度与最终精度

RMSProp的一大核心优势在于其**收敛速度极快**。在大量基准测试与深度学习任务中，它能够迅速降低训练损失，并在相对较少的迭代步数内达到较高的最终精度。

- **典型实验数据**：有研究表明，在标准图像分类任务中，RMSProp的平均测试准确率可达**68.11%**，在部分配置下甚至能超过**90%**。
- **机制解释**：这一表现主要得益于其自适应的学习率调整机制——通过为每个参数独立维护近期梯度的均方根统计量，RMSProp能够有效应对复杂、非凸的目标函数（如RNN/LSTM的训练），避免因学习率过早衰减而陷入次优解。

#### C.2 稳定性与泛化能力

这是RMSProp最具争议性的维度——其表现**高度依赖于具体任务与数据分布**，无法一概而论。

- **在特定任务中表现出色**：部分研究发现，RMSProp不仅精度高，而且多次运行的结果方差较小（即稳定性好），这意味着它在这些场景下是一个可靠的选择。
- **在另一些任务中则不稳定**：也有实验表明，RMSProp的验证损失和准确率会出现大幅波动，其泛化能力（即在未见测试集上的表现）可能不如Adam或带有权重衰减的GD。例如，在某梵文字符识别任务中，它的准确率仅为**81.74%**，在对比的若干优化器中排名靠后。

#### C.3 性能背后的原因

- **优势来源**：RMSProp有效解决了AdaGrad学习率过早且不可逆衰减的致命缺陷。它使用指数加权移动平均替代历史累加，使得统计量能够随地形变化而动态调整，从而在训练全程保持持续学习的能力。理论上，其更新方向也趋向于与标准梯度下降的方向一致，这为它的优异性能提供了部分理论支撑。
- **劣势来源**：RMSProp的最终性能对超参数（尤其是初始学习率$\eta$和衰减率$\beta$）较为敏感，需要仔细调参。此外，它不包含动量项，在极端陡峭或存在剧烈梯度震荡的地形中，可能表现出不必要的震荡，影响收敛的平稳性。

#### C.4 结论性评价

RMSProp的最终性能**没有一个统一的、放之四海而皆准的答案**。它是一个能力极强的优化器，尤其适合需要快速收敛和对梯度动态变化敏感的任务（如循环神经网络）。但它能否在某个特定问题上达到最优，通常需要与Adam、GD等优化器进行实际的对比实验后才能确定。在深度学习实践中，它依然是一个值得纳入候选列表的重要选项。


### 附录D. RMSProp的启发式本质与理论证明辨析

本附录针对RMSProp是否属于启发式算法、其性能有无严格理论证明这一问题，给出严谨的数学与哲学层面的辨析。

#### D.1 严格意义上的结论：RMSProp是启发式算法

从**严格数学意义上**讲，RMSProp确实属于**启发式算法（Heuristic Algorithm）**，其最优秀的性能表现缺乏严格的理论证明。

在优化算法理论中，"启发式"意味着算法的设计主要基于**直觉、经验或观察到的现象**，而不是从一个严格的数学问题（如"最小化某个确定性的凸函数"）出发推导出来的。

RMSProp的设计动机完全是启发式的：

- **问题来源**：研究者观察到AdaGrad的学习率"只降不升"，在训练后期梯度平方累加项$G$无限膨胀，导致有效步长趋近于零。
- **修补方案**：直观上，如果让历史信息"衰减"掉，只关注近期梯度变化，就能避免这一问题。于是，研究者"拍脑袋"提出了**指数加权移动平均（EMA）** 这一经验性操作。
- **结论**：这个操作虽然在实践中效果拔群，但它并非从"严格最小化凸函数"或"严格保证最坏情况收敛速度"等数学框架下推导出的必然结果。它就是一个基于**实践观察**和**工程直觉**的修补方案——这正是启发式算法的典型特征。

#### D.2 "有"与"没有"：收敛性证明 vs. 最优性证明

这需要严谨地区分两个不同层次的理论问题：

1. **有"收敛性"证明（弱保证）**：
   - 对于**凸函数**，研究者确实证明了RMSProp在特定条件下可以收敛到最优解。
   - **但需要注意**：这个证明对学习率的设置、超参数有非常苛刻的限制——例如，学习率必须按照特定的调度策略逐渐衰减到零。
   - **关键点**：在实际使用中，我们常用的那种"固定大学习率 + 默认参数"的RMSProp，**并不在这个理论证明的范围内**。

2. **没有"最优性"证明（强保证）**：
   - 对于**非凸函数**（深度学习的核心战场），目前没有任何理论能证明RMSProp一定比GD、Adam或其他算法更好，更无法证明它能找到全局最优解。
   - 它的"最终性能"（例如在某任务上达到90%的准确率）完全依赖实际运行结果，无法像线性回归那样通过数学公式预先计算出最优解。

#### D.3 为什么实践中仍广泛使用？

尽管理论证明缺失，RMSProp在工业界和学术界依然占据重要地位，原因在于**深度学习的理论严重滞后于实践**。

- **强大的实证证据（Empirical Evidence）**：虽然缺乏"为什么这么强"的理论依据，但RMSProp拥有海量的成功案例。在2012—2016年间，它在训练RNN、LSTM等对梯度缩放敏感的模型时，成功率远高于调参困难的标准GD，这使它成为了那个时期的"事实标准"。
- **类比**：这类似于某些传统医学中的有效方剂——虽然现代科学尚未完全解析其分子机制（理论缺失），但大量的临床数据（实验结果）证明其确实有效。

#### D.4 总结

> **RMSProp是一个"实验上极其成功，但理论上尚不完善"的启发式算法。**

它在非凸问题上的"最终性能"，目前依然要靠**实际运行实验来确定**——这也是为什么深度学习工程师们至今仍然会在RMSProp、Adam、GD等优化器之间进行网格搜索或交叉验证：因为没有人能提前从数学上算出哪一个在你的特定数据和模型上表现最佳。

**延伸思考**：如果认为"没有理论证明"让人心里没底，那么当前的主流趋势（如AdamW）正是在试图将启发式算法的优势与更严谨的权重衰减等理论修正结合起来，向着"既有实证效果、又有更清晰理论解释"的方向演进。